# Сверка agr_id из Excel с Озером за 3 месяца

Что делает тетрадка:
1. Читает Excel-отчеты за январь, февраль и март.
2. Нормализует `ИНН` и `agr_id`.
3. Сравнивает `agr_id` из Excel с `ods_alpha.scd1_agreements.abs_agr_id` (`acq_class = 'SA'`).
4. Выводит помесячную сводку, общий итог и 5 примеров ИНН по каждому месяцу.

In [ ]:
import re
from decimal import Decimal, InvalidOperation
import pandas as pd
from rail_connectors.connection import connect
from connection_secrets import LAKE_USER, LAKE_PASSWORD

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры
sources = [
    {"report_month": "2026-01", "excel_path": "/home/jovyan/documents/Equaring/Data/01_Январь_2026.xlsx", "header": 1},
    {"report_month": "2026-02", "excel_path": "/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx", "header": 0},
    {"report_month": "2026-03", "excel_path": "/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx", "header": 0},
]

MEM_LIMIT = "8g"

# Кандидаты названий колонок в Excel
INN_COL_CANDIDATES = ["ИНН", "inn", "c_inn", "client_inn"]
AGR_COL_CANDIDATES = ["ID договора", "agr_id", "id_dogovora", "abs_agr_id", "cdi_id"]

In [ ]:
def normalize_num_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(" ", "").replace("\xa0", "")
    if s == "" or s.lower() in {"nan", "none", "null"}:
        return None
    s = s.replace(",", ".")
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), "f")
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", "", s)
    s = re.sub(r"\D", "", s)
    return s if s else None

def pick_col(df, candidates):
    norm_map = {str(c).strip().lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in norm_map:
            return norm_map[c.lower()]
    return None

## 1) Загружаем Excel за 3 месяца и нормализуем INN / agr_id

In [ ]:
excel_rows = []

for src in sources:
    df = pd.read_excel(src["excel_path"], header=src.get("header", 0))
    df.columns = [str(c).strip() for c in df.columns]

    inn_col = pick_col(df, INN_COL_CANDIDATES)
    agr_col = pick_col(df, AGR_COL_CANDIDATES)

    if not inn_col or not agr_col:
        raise ValueError(
            f"[{src['report_month']}] Не найдены нужные колонки: inn_col={inn_col}, agr_col={agr_col}."
        )

    tmp = df[[inn_col, agr_col]].copy()
    tmp.columns = ["inn_raw", "agr_raw"]
    tmp["inn_norm"] = tmp["inn_raw"].apply(normalize_num_str)
    tmp["agr_id_norm"] = tmp["agr_raw"].apply(normalize_num_str)
    tmp["report_month"] = src["report_month"]

    # сравниваем только строки, где в Excel есть agr_id
    tmp = tmp[tmp["agr_id_norm"].notna()].copy()
    excel_rows.append(tmp[["report_month", "inn_norm", "agr_id_norm"]])

excel_all = pd.concat(excel_rows, ignore_index=True).drop_duplicates()

print("excel_rows_total =", len(excel_all))
print("excel_agr_id_distinct_total =", excel_all["agr_id_norm"].nunique())
display(excel_all.head(20))

## 2) Сравниваем с Озером и строим сводку + 5 примеров ИНН

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': LAKE_USER, 'password': LAKE_PASSWORD}
)
imp._init_connection()
print('Impala connection initialized')

with imp:
    imp.execute(f"SET MEM_LIMIT={MEM_LIMIT}")
    lake_agr = imp.fetch("""
        select distinct cast(a.abs_agr_id as string) as agr_id_norm
        from ods_alpha.scd1_agreements a
        where a.abs_agr_id is not null
          and a.acq_class = 'SA'
    """)

lake_agr['agr_id_norm'] = lake_agr['agr_id_norm'].apply(normalize_num_str)
lake_agr = lake_agr[lake_agr['agr_id_norm'].notna()].drop_duplicates()
lake_agr_set = set(lake_agr['agr_id_norm'].tolist())

excel_all['missing_in_lake'] = ~excel_all['agr_id_norm'].isin(lake_agr_set)
missing_df = excel_all[excel_all['missing_in_lake']].copy()

monthly_summary = (
    missing_df.groupby('report_month', as_index=False)
    .agg(
        missing_agr_id_cnt=('agr_id_norm', 'nunique'),
        missing_inn_cnt=('inn_norm', 'nunique'),
        missing_rows_cnt=('agr_id_norm', 'size')
    )
    .sort_values('report_month')
)

total_summary = pd.DataFrame([{
    'report_month': 'TOTAL_2026Q1',
    'missing_agr_id_cnt': missing_df['agr_id_norm'].nunique(),
    'missing_inn_cnt': missing_df['inn_norm'].nunique(),
    'missing_rows_cnt': len(missing_df)
}])

summary = pd.concat([monthly_summary, total_summary], ignore_index=True)
print('Сводка расхождений (agr_id есть в Excel, но нет в Озере):')
display(summary)

examples_5_inn = (
    missing_df[missing_df['inn_norm'].notna()]
    .sort_values(['report_month', 'inn_norm', 'agr_id_norm'])
    .groupby('report_month', as_index=False)
    .head(5)[['report_month', 'inn_norm', 'agr_id_norm']]
)

print('5 примеров ИНН по каждому месяцу:')
display(examples_5_inn)

# при необходимости: выгрузка результатов
# summary.to_excel('/home/jovyan/documents/Equaring/Проверки/agr_id_missing_summary_3months.xlsx', index=False)
# examples_5_inn.to_excel('/home/jovyan/documents/Equaring/Проверки/agr_id_missing_examples_5inn_per_month.xlsx', index=False)
# missing_df.to_excel('/home/jovyan/documents/Equaring/Проверки/agr_id_missing_full_3months.xlsx', index=False)